# Phase 16 — Pipeline Validation (SQL + NoSQL combined)

Runs the new `src/pipeline_sql.py` / `src/pipeline_nosql.py` library code (Phase 16)
end-to-end on whatever GPU this session gets — **works unmodified on T4 or A100**.
`GeneratorInfer` (`src/generator/infer.py`) loads the 7B model in int8 (bitsandbytes)
on any GPU under 24GB so it fits entirely in VRAM on a T4, and full bf16 on A100-class
(≥24GB) GPUs where it fits without quantization — see the comments in that file from
the Phase 14/15 OOM and CPU-offload fixes. No code changes needed to switch GPU type;
**T4 will be a bit slower** (int8 vs bf16) than A100, not offloaded to CPU and hung.

Two things this notebook checks that `Phase15_POSG_Combined.ipynb` didn't:
1. **Direct calls into `src.pipeline_sql.run_pipeline()` / `src.pipeline_nosql.run_pipeline()`**
   (not just the `scripts/run_posg_*.py` CLI) — confirms the new Phase 16 library
   functions work standalone, since that's what Phase 17's LangGraph router will call.
2. **Which GPU this session actually got**, printed up front, so you know whether to
   expect A100-speed (bf16) or T4-speed (int8) generation before waiting on anything.

See `docs/phase15_posg_findings.md` for the full Phase 15 methodology and EX results
this notebook's smoke-test cells reproduce.

## 0. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"GPU {i}: {name}  ({total_gb:.1f} GB)")
        if total_gb >= 24:
            print("  -> >=24GB (A100-class, ~15 CU/hr). GeneratorInfer loads full bf16 -- "
                  "fastest path, but this notebook is inference-only (single questions + "
                  "n=30 smoke tests) and doesn't need A100 headroom. T4 covers it at "
                  "~1.9 CU/hr (~8x cheaper); consider Runtime > Change runtime type > T4 "
                  "GPU unless you specifically want the speed.")
        else:
            print("  -> <24GB (T4-class, ~1.9 CU/hr): right choice for this notebook. "
                  "GeneratorInfer loads the 7B model in int8 (bitsandbytes) so it fits "
                  "entirely in VRAM -- no CPU offload, no hang. See src/generator/infer.py.")
else:
    print("No GPU detected -- runtime will fall back to CPU, which is impractically slow "
          "for a 7B model doing 5-candidate generation. Set Runtime > Change runtime type > GPU.")

## 1. Clone repo + install dependencies

In [ ]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo bitsandbytes

## 2. Mount Drive and load checkpoints (both tracks)

Loads SAR + Generator checkpoints for **both** SQL and NoSQL from Drive. If your Drive layout differs from `codegen/checkpoints/{sar,generator}_{sql,nosql}`, adjust `DRIVE` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

for name in ['sar_sql', 'generator_sql', 'sar_nosql', 'generator_nosql']:
    dst = f'models/{name}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/checkpoints/{name}', dst)

!ls -la models/sar_sql models/generator_sql models/sar_nosql models/generator_nosql

In [ ]:
# Safety net: force sar.backend to memory. ChromaDB's PersistentClient can't open
# an index over a Google Drive FUSE mount, so this avoids that failure mode entirely.
text = open('configs/config.yaml').read()
text = text.replace('backend: chroma', 'backend: memory')
open('configs/config.yaml', 'w').write(text)
!grep -A1 "^sar:" configs/config.yaml | head -3

## 3. Spider SQLite databases

Needed for SQL EX scoring, and as the source data for the MongoDB conversion below. Uploaded once as a zip to Drive (see `docs/phase15_posg_findings.md` for how it was built, from a sibling local project).

In [ ]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l

## 4. MongoDB setup (needed for NoSQL EX)

**Important**: `Data/mongodb/*.json` schema-cache files are git-tracked from an earlier run on a different machine. `convert_all()` treats their existence as "already converted" and will silently skip real data insertion into this fresh session's empty `mongod` if we don't clear them first (see `docs/phase15_posg_findings.md`).

In [ ]:
# Install and start mongod
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess, time
subprocess.Popen(["mongod", "--dbpath", "/data/db", "--bind_ip", "127.0.0.1"])
time.sleep(5)
!mongosh --eval "db.version()" 

In [ ]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)   # force a real reconversion, see note above

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)

In [ ]:
# Verify real data landed (not just schema cache) before trusting any EX result
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(len(client.list_database_names()), client.list_database_names()[:10])
print("formula_1.drivers count:", client['formula_1']['drivers'].count_documents({}))   # must be > 0

## 5. SQL track — direct `run_pipeline()` call (Phase 16 library test)

Calls `src.pipeline_sql.run_pipeline()` directly in-process (not via the CLI script) — this
is exactly what Phase 17's LangGraph router will do, so a clean run here is the real Phase 16
validation. Loads `configs/config.yaml`, builds the schema for `concert_singer` internally,
runs SchemaLinker -> SAR -> Generator (5 candidates) -> POSG, and returns the selected SQL.

In [ ]:
import yaml
from src.pipeline_sql import run_pipeline

config = yaml.safe_load(open("configs/config.yaml"))

result = run_pipeline(
    question="How many singers are there?",
    db_name="concert_singer",
    config=config,
    strategy="balanced",
)

print("candidates:")
for c in result["candidates"]:
    print(" -", c)
print("\ngreedy:  ", result["greedy"])
print("selected:", result["selected"])
print("posg_diverged:", result["posg_diverged"])

## 6. SQL track — full dev-set smoke test (all difficulty, Phase 15 baseline)

`sql_dev_eval_full.json` (1034 Spider dev questions, real SchemaLinker `key_fields`) was rebuilt locally on Mac after the Colab runtime that originally built it disconnected. Upload it to Drive from your Mac first, then pull it in here. This run (no `--hard` filter) is the one comparable to the plan's >82% EX target -- reproduces the Phase 15A numbers using the (now-refactored) CLI, doubling as a regression check that the Phase 16 extraction didn't change behavior.

In [ ]:
%%time
# Timed so you can extrapolate to the full 1034-question dev set before
# committing compute-unit budget to it: (this cell's wall time / 30) * 1034.
!cp /content/drive/MyDrive/codegen/sql_dev_eval_full.json Data/cot_data/sql_dev_eval_full.json
!python -m scripts.run_posg_sql --smoke_test --n 30 --data Data/cot_data/sql_dev_eval_full.json

## 7. NoSQL track — direct `run_pipeline()` call (Phase 16 library test)

Same idea as section 5, calling `src.pipeline_nosql.run_pipeline()` directly. Requires the live `mongod` + loaded databases from section 4.

In [ ]:
from src.pipeline_nosql import run_pipeline as run_pipeline_nosql

result_nosql = run_pipeline_nosql(
    question="How many singers are there?",
    db_name="concert_singer",
    config=config,
    strategy="balanced",
)

import json
print("candidates:")
for c in result_nosql["candidates"]:
    print(" -", json.dumps(c, ensure_ascii=False))
print("\ngreedy:  ", json.dumps(result_nosql["greedy"], ensure_ascii=False))
print("selected:", json.dumps(result_nosql["selected"], ensure_ascii=False))
print("posg_diverged:", result_nosql["posg_diverged"])

## 8. NoSQL track — train-split smoke test (all difficulty, Phase 15 baseline)

Already validated: EX 76.7% (POSG) vs 73.3% (greedy) on this exact command (Phase 15B). Re-run here to confirm reproducibility after the Phase 16 refactor.

In [ ]:
%%time
!python -m scripts.run_posg_nosql --smoke_test --n 30

## 9. Optional — DeepSeek API key

Only needed for the `SchemaLinker` calls in sections 5/7 above (`run_pipeline()` calls it live) or to rebuild a dev-eval-set file. The smoke tests in sections 6/8 reuse pre-computed `key_fields` and never call DeepSeek. **Never commit this cell with a real key filled in.**

In [ ]:
with open('.env', 'w') as f:
    f.write('DEEPSEEK_API_KEY=your_actual_key_here\n')

## 10. Free the GPU when you're done

Units keep burning as long as the runtime stays connected, even idle. Run
this once you've read the output above and don't need the session anymore —
it disconnects and releases the GPU. (Not run automatically: you may still
want to inspect variables or re-run a cell first.)

In [ ]:
from google.colab import runtime
runtime.unassign()